# Beyond statistical significance: interpretation and reporting of p values and confidence intervals with computational implementation in r and python
**Authors:** Renato Carneiro de Freitas Chaves, Tiago Mendonça dos Santos, Thiago Domingos Corrêa  

## 1) Purpose


This notebook provides a step-by-step analytical structure for **clinical research** comparing:
- **Control group** (`group = 0`)
- **Intervention group** (`group = 1`)

Using the attached dataset, it demonstrates how to:
1. Formulate **null and alternative hypotheses** for common epidemiological comparisons.
2. Choose and run **hypothesis tests** (with decision rules using **α**).
3. Compute and interpret **p values**.
4. Compute and interpret **95% confidence intervals (CIs)**, and connect CIs to hypothesis testing.

The example uses a **binary outcome** (`outcome`) compared between:
- **Control group** (`group = 0`)
- **Intervention group** (`group = 1`)

A secondary example illustrates hypothesis testing and CIs for a **continuous variable** (e.g., `age`) across groups.

> **Dataset coding (per the model data file)**  
> - `group`: `0 = Control`, `1 = Intervention`  
> - `outcome`: `0 = Alive`, `1 = Dead`  

    > The **event of interest** is **death** (`outcome = 1`).


## 2) Required Libraries

This notebook uses standard scientific Python libraries.

In [43]:
import numpy as np
import pandas as pd
from scipy import stats
from scipy.stats import chi2_contingency, fisher_exact

# Display options (optional)
pd.set_option("display.max_rows", 200)
pd.set_option("display.max_columns", 50)

## 3) Data Import (Excel)

This guide uses the Excel file **`data.xlsx`** with a sheet named **`Data`**.

- If `data.xlsx` is in the same folder as this notebook, you can keep the path as `"data.xlsx"`.
- Otherwise, replace with the full path to your file.


In [44]:
# Update the path if needed
excel_path = "data.xlsx"
sheet_name = "Data"

data = pd.read_excel(excel_path, sheet_name=sheet_name)

# Quick inspection
data.head(), data.shape

(   patient  group  center  age  platelet  hemoglobin  creatinine  \
 0        1      1       1   19       229        15.3         0.8   
 1        2      1       1   24       221        14.8         0.8   
 2        3      1       1   25       219        14.4         0.9   
 3        4      1       1   26       217        14.3         0.9   
 4        5      1       1   27       214        13.8         0.9   
 
    hypertension  diabetes  heart_failure  smoking  chronic_kidney_disease  \
 0             0         0              0        0                       0   
 1             0         0              0        0                       0   
 2             0         0              0        0                       0   
 3             0         0              0        0                       0   
 4             1         0              1        0                       1   
 
    cancer  stroke  saps_3  vasopressor  mechanical_ventilation   \
 0       0       0       5            0       

In [45]:
# Column names and basic info

data.info()

list(data.columns)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 50 entries, 0 to 49
Data columns (total 19 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   patient                  50 non-null     int64  
 1   group                    50 non-null     int64  
 2   center                   50 non-null     int64  
 3   age                      50 non-null     int64  
 4   platelet                 50 non-null     int64  
 5   hemoglobin               50 non-null     float64
 6   creatinine               50 non-null     float64
 7   hypertension             50 non-null     int64  
 8   diabetes                 50 non-null     int64  
 9   heart_failure            50 non-null     int64  
 10  smoking                  50 non-null     int64  
 11  chronic_kidney_disease   50 non-null     int64  
 12  cancer                   50 non-null     int64  
 13  stroke                   50 non-null     int64  
 14  saps_3                   50 

['patient',
 'group',
 'center',
 'age',
 'platelet',
 'hemoglobin',
 'creatinine',
 'hypertension',
 'diabetes',
 'heart_failure',
 'smoking',
 'chronic_kidney_disease',
 'cancer',
 'stroke',
 'saps_3',
 'vasopressor',
 'mechanical_ventilation ',
 'fluid_balance',
 'outcome']

## 4) Data Validation and Preparation

In [46]:
required_cols = {"group", "outcome"}
missing = required_cols - set(data.columns)

if missing:
    raise ValueError(f"Missing required column(s): {sorted(missing)}")

# Frequency checks
data["group"].value_counts(dropna=False), data["outcome"].value_counts(dropna=False)

(1    25
 0    25
 Name: group, dtype: int64,
 1    30
 0    20
 Name: outcome, dtype: int64)

In [47]:
def format_p(p, digits=3):
    if pd.isna(p):
        return np.nan
    return "< 0.001" if p < 0.001 else f"{p:.{digits}f}"


## 5) Create the Analysis Dataset

We will:

- keep only rows with non-missing `group` **and** `outcome`
- ensure they are coded as expected (0/1)
- define the event as `outcome == 1` (death)


In [48]:
analysis_data = (
    data.loc[:, ["group", "outcome"]]
        .dropna(subset=["group", "outcome"])
        .assign(
            group=lambda df: df["group"].astype(int),
            event=lambda df: df["outcome"].astype(int)  # event = 1 means "death"
        )
)

analysis_data.head(), analysis_data.shape

(   group  outcome  event
 0      1        0      0
 1      1        0      0
 2      1        0      0
 3      1        0      0
 4      1        0      0,
 (50, 3))

## 6) Construct the 2×2 Table

Rows:
- Intervention
- Control

Columns:
- Event (Dead)
- Non-event (Alive)

In [49]:
analysis_data = analysis_data.assign(
    group_label=pd.Categorical(
        analysis_data["group"],
        categories=[1, 0],
        ordered=True
    ).rename_categories(["Intervention", "Control"]),
    event_label=pd.Categorical(
        analysis_data["event"],
        categories=[1, 0],
        ordered=True
    ).rename_categories(["Event", "Non-event"])
)

tab = pd.crosstab(analysis_data["group_label"], analysis_data["event_label"])
tab

event_label,Event,Non-event
group_label,,
Intervention,11,14
Control,19,6


## 7) Hypothesis Testing for the Binary Outcome

### Null Hypothesis (H0)
There is no association between `group` and `outcome`.

### Alternative Hypothesis (H1)
The event risk differs between groups.

### Significance Level
α = 0.05


In [50]:
alpha = 0.05
alpha

0.05

### 7.1) Expected Counts (Chi-square Assumptions)

In [51]:
chi2_stat, chi2_p, dof, expected = chi2_contingency(tab.values, correction=False)
pd.DataFrame(expected, index=tab.index, columns=tab.columns)


event_label,Event,Non-event
group_label,,
Intervention,15.0,10.0
Control,15.0,10.0


In [52]:
oddsratio, fisher_p = fisher_exact(tab.values)

use_fisher = (expected < 5).any()
test_recommended = "Fisher's exact test" if use_fisher else "Chi-square test"

pd.DataFrame({
    "Recommended Test": [test_recommended],
    "Chi-square p-value": [chi2_p],
    "Fisher p-value": [fisher_p]
})


,Recommended Test,Chi-square p-value,Fisher p-value
0,Chi-square test,0.020921,0.042078


### 7.2) Decision Rule

In [53]:
p_used = fisher_p if use_fisher else chi2_p
decision = "Reject H0" if p_used <= alpha else "Fail to reject H0"

pd.DataFrame({
    "Alpha": [alpha],
    "P-value used": [p_used],
    "Decision": [decision]
})


,Alpha,P-value used,Decision
0,0.05,0.020921,Reject H0


## 8) Continuous Variable Example (Age)

In [54]:
if "age" not in data.columns:
    raise ValueError("The dataset must include an 'age' column.")

age_data = data.loc[:, ["group", "age"]].dropna()

age_int = age_data[age_data["group"] == 1]["age"].astype(float)
age_ctl = age_data[age_data["group"] == 0]["age"].astype(float)

age_data.groupby("group")["age"].agg(["count", "mean", "std"])


,count,mean,std
group,,,
0,25,61.24,11.896358
1,25,31.20,6.467869


### 8.1) Welch Two-Sample t-Test + 95% CI

In [55]:
tt = stats.ttest_ind(age_int, age_ctl, equal_var=False)
p_value_age = tt.pvalue

mean_difference = age_int.mean() - age_ctl.mean()

n1, n0 = len(age_int), len(age_ctl)
s1, s0 = age_int.std(ddof=1), age_ctl.std(ddof=1)

se = np.sqrt((s1**2 / n1) + (s0**2 / n0))

df = (s1**2 / n1 + s0**2 / n0) ** 2 / (
    ((s1**2 / n1) ** 2) / (n1 - 1) + ((s0**2 / n0) ** 2) / (n0 - 1)
)

tcrit = stats.t.ppf(1 - alpha / 2, df)

CI_low = mean_difference - tcrit * se
CI_high = mean_difference + tcrit * se

pd.DataFrame({
    "P-value": [p_value_age],
    "Mean difference (Intervention - Control)": [mean_difference],
    "95% CI lower": [CI_low],
    "95% CI upper": [CI_high],
    "Degrees of freedom (Welch)": [df]
})


,P-value,Mean difference (Intervention - Control),95% CI lower,95% CI upper,Degrees of freedom (Welch)
0,2.481637e-13,-30.04,-35.527062,-24.552938,37.048368


## 9) Final Summary Table

In [61]:
# A compact, publication table with CI columns where applicable
final_table = pd.DataFrame([
    {"measure": "Recommended test for group vs outcome", "value": test_recommended, "lower_ci": np.nan, "upper_ci": np.nan},
    {"measure": "P value (Chi-square); group vs outcome", "value": format_p(chi2_p), "lower_ci": np.nan, "upper_ci": np.nan},
    {"measure": "P value (Fisher); group vs outcome", "value": format_p(fisher_p), "lower_ci": np.nan, "upper_ci": np.nan},
    {"measure": "Alpha (α); group vs outcome", "value": f"{alpha:.2f}", "lower_ci": np.nan, "upper_ci": np.nan},
    {"measure": "Decision at α=0.05; group vs outcome", "value": decision, "lower_ci": np.nan, "upper_ci": np.nan},
    {"measure": "P value for age (group comparison)", "value": format_p(p_value_age), "lower_ci": np.nan, "upper_ci": np.nan},
    {"measure": "Age mean difference (group comparison)", "value": f"{mean_difference:.2f}", "lower_ci": f"{CI_low:.2f}", "upper_ci": f"{CI_high:.2f}"},
])

final_table.round(3)

,measure,value,lower_ci,upper_ci
0,Recommended test for group vs outcome,Chi-square test,NaN,NaN
1,P value (Chi-square); group vs outcome,0.021,NaN,NaN
2,P value (Fisher); group vs outcome,0.042,NaN,NaN
3,Alpha (α); group vs outcome,0.05,NaN,NaN
4,Decision at α=0.05; group vs outcome,Reject H0,NaN,NaN
5,P value for age (group comparison),< 0.001,NaN,NaN
6,Age mean difference (group comparison),-30.04,-35.53,-24.55
